# Notebook 05 — Training a Reward Model

A reward model learns to score LLM outputs by preference — it is the backbone of RLHF. We train a binary preference classifier: given two responses, predict which one humans prefer.

In [ ]:
# !pip install transformers datasets torch accelerate

## 1. What is a reward model?

In [ ]:
# A reward model takes (prompt, response) pairs and outputs a scalar score.
# Training data: pairs of responses where annotators marked one as preferred.
#
# Architecture: a language model with a regression head instead of an LM head.
# Training loss: Bradley-Terry pairwise ranking loss
#   L = -log(sigmoid(score_chosen - score_rejected))

import torch
import torch.nn.functional as F

def pairwise_loss(score_chosen: torch.Tensor, score_rejected: torch.Tensor) -> torch.Tensor:
    return -F.logsigmoid(score_chosen - score_rejected).mean()

# Demo
chosen  = torch.tensor([2.5, 1.8, 3.1])
rejected = torch.tensor([1.0, 0.5, 2.0])
print("Pairwise loss:", pairwise_loss(chosen, rejected).item())

## 2. Load preference data

In [ ]:
import json, os

# Use the sample data file shipped with this repo
data_path = "../data/preference_examples.jsonl"
examples = []
if os.path.exists(data_path):
    with open(data_path) as f:
        for line in f:
            examples.append(json.loads(line))
else:
    # Inline fallback
    examples = [
        {"prompt": "Explain quantum computing",
         "chosen": "Quantum computers use qubits that can represent 0 and 1 simultaneously.",
         "rejected": "It uses quantum stuff to compute fast."},
        {"prompt": "What is gradient descent?",
         "chosen": "Gradient descent minimises a loss function by iteratively moving in the direction of steepest descent.",
         "rejected": "It makes the model learn by going down."},
    ]

print(f"Loaded {len(examples)} preference pairs")
print(examples[0])

## 3. Build the reward model

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
reward_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=1)
print(reward_model.config)

## 4. Training loop (demonstration)

In [ ]:
import torch
from torch.optim import AdamW

optimizer = AdamW(reward_model.parameters(), lr=2e-5)
reward_model.train()

for epoch in range(2):
    total_loss = 0
    for ex in examples:
        chosen_enc  = tokenizer(ex["prompt"], ex["chosen"],  return_tensors="pt", truncation=True, max_length=128)
        rejected_enc = tokenizer(ex["prompt"], ex["rejected"], return_tensors="pt", truncation=True, max_length=128)

        score_chosen   = reward_model(**chosen_enc).logits.squeeze()
        score_rejected = reward_model(**rejected_enc).logits.squeeze()

        loss = pairwise_loss(score_chosen.unsqueeze(0), score_rejected.unsqueeze(0))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch + 1} loss: {total_loss / len(examples):.4f}")

## 5. Score new responses

In [ ]:
reward_model.eval()
with torch.no_grad():
    for text in ["This is a clear, accurate answer.", "Idk lol"]:
        enc = tokenizer("Test question", text, return_tensors="pt", truncation=True)
        score = reward_model(**enc).logits.item()
        print(f"Score={score:.3f}  |  {text}")